In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
import re
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans

In [2]:
# 1. Load the Data from Hugging Face
print("Loading dataset...")
# This pulls the dataset directly. If you downloaded the CSV manually,
# you can just use df = pd.read_csv('your_file.csv')
dataset = load_dataset("ganchengguang/resume_seven_class", split="train")
df = pd.DataFrame(dataset)

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:
df.head(20)

,text,cleaned_resume,assigned_cluster,true_label,mapped_prediction
0,Exp\tName: Abiral Pandey,exp name abiral pandey,6,exp,exp
1,PI\tEmail:,pi email,1,pi,pi
2,PI\tPhone: 940-242-3303,pi phone,1,pi,pi
3,"PI\tCurrent Location: Woonsocket, Rhode Island",pi current location woonsocket rhode island,1,pi,pi
4,PI\tVisa Status: US Citizen,pi visa status us citizen,1,pi,pi
5,Sum\tSUMMARY:,sum summary,5,sum,sum
6,Sum\tDynamic individual with 6 years of softwa...,sum dynamic individual with years of software ...,2,sum,exp
7,Sum\tExposure to all phases of Software Develo...,sum exposure to all phases of software develop...,5,sum,sum
8,Sum\tDesigned and developed web UI screen usin...,sum designed and developed web ui screen using...,2,sum,exp
9,"Sum\tDeveloped AngularJS Controllers, Services...",sum developed angularjs controllers services f...,2,sum,exp


In [11]:
# 2. Preprocessing Function
def advanced_clean_text(text):
    text = str(text)

    # 1. Strip out the structural tags (e.g., "PI\tPhone:", "Exp\tName:")
    # This regex looks for a word, a literal '\t', optional words, and a colon.
    text = re.sub(r'[A-Za-z]+\\t[A-Za-z\s]*:', ' ', text)

    # Also catch any lingering tags that don't have a colon (like "PI\t")
    text = re.sub(r'[A-Za-z]+\\t', ' ', text)

    # 2. Remove Phone Numbers (e.g., 940-242-3303)
    text = re.sub(r'\d{3}-\d{3}-\d{4}', ' ', text)

    # 3. Remove Emails
    text = re.sub(r'\S+@\S+', ' ', text)

    # 4. Standard NLP Cleaning (Lowercase, keep only letters, remove extra spaces)
    text = text.lower()
    text = re.sub(r'[^a-z]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

print("Applying advanced cleaning...")
df['cleaned_resume'] = df['text'].apply(advanced_clean_text)

Applying advanced cleaning...


In [12]:
print("Initializing TF-IDF Vectorizer...")
# max_features=1500 limits the algorithm to the top 1500 most important words.
# This prevents our matrix from becoming millions of columns wide and crashing your computer.
# stop_words='english' is a safety net to remove any lingering common words like "the", "is", "at".
vectorizer = TfidfVectorizer(max_features=1500, stop_words='english')

print("Learning vocabulary and converting text to numbers...")
# .fit_transform() does the heavy lifting:
# 1. It scans all resumes to find the 1500 best words.
# 2. It scores every resume based on those words.
X = vectorizer.fit_transform(df['cleaned_resume'])

print(f"Success! The mathematical matrix shape is: {X.shape}")

Initializing TF-IDF Vectorizer...
Learning vocabulary and converting text to numbers...
Success! The mathematical matrix shape is: (78670, 1500)


In [13]:
print("Running K-Means Clustering (dropping the 7 anchors)...")
# MiniBatch is much faster for large datasets like yours.
# n_clusters=7 because we know there are 7 real categories in this dataset.
kmeans = MiniBatchKMeans(n_clusters=7, random_state=42, batch_size=2048, n_init="auto")

# Train the model on your matrix!
kmeans.fit(X)

# Tag each original resume with its new cluster number (0 through 6)
df['assigned_cluster'] = kmeans.labels_
print("Clustering complete!\n")

print("--- Algorithm's Discovered Clusters ---")
# Get the actual English words back from the TF-IDF vectorizer
feature_names = vectorizer.get_feature_names_out()
centroids = kmeans.cluster_centers_

# Loop through all 7 clusters and print the top 10 defining words for each
for i in range(7):
    print(f"\nCluster {i} Top Keywords:")
    # Sort the centroid weights to find the most important words
    top_indices = centroids[i].argsort()[-10:][::-1]
    top_words = [feature_names[ind] for ind in top_indices]
    print(", ".join(top_words))

Running K-Means Clustering (dropping the 7 anchors)...
Clustering complete!

--- Algorithm's Discovered Clusters ---

Cluster 0 Top Keywords:
data, exp, reports, sql, database, using, procedures, queries, stored, analysis

Cluster 1 Top Keywords:
pi, date, personal, birth, mobile, nationality, indian, place, email, father

Cluster 2 Top Keywords:
using, exp, used, web, spring, application, developed, java, html, services

Cluster 3 Top Keywords:
skill, obj, skills, objective, declaration, knowledge, technical, ms, career, windows

Cluster 4 Top Keywords:
edu, university, board, education, college, school, year, qualification, com, educational

Cluster 5 Top Keywords:
sum, experience, work, summary, professional, good, skills, exp, years, achievements

Cluster 6 Top Keywords:
exp, project, responsibilities, team, client, business, qc, management, role, duration


In [14]:
print("1. Extracting the hidden true labels...")
# This regex splits the text at the first tab (\t) or space, grabs the first piece, and lowercases it.
df['true_label'] = df['text'].apply(lambda x: re.split(r'[\t\s]', str(x))[0].lower())

# Let's see the 7 actual classes the dataset author intended!
print(f"Discovered Classes: {df['true_label'].unique()}\n")

# 2. Set our new column as the ground truth
true_label_column = 'true_label'

print("--- The Overlap Grid ---")
crosstab = pd.crosstab(df[true_label_column], df['assigned_cluster'])
print(crosstab)

print("\n--- Mapping Clusters to True Labels ---")
cluster_to_label_map = {}

for cluster_id in range(7):
    # Look at resumes in the current cluster
    cluster_resumes = df[df['assigned_cluster'] == cluster_id]

    # Prevent errors if a cluster happens to be totally empty
    if not cluster_resumes.empty:
        # Find the most frequent true label (e.g., 'pi', 'edu') inside this cluster
        most_common_label = cluster_resumes[true_label_column].mode()[0]
        cluster_to_label_map[cluster_id] = most_common_label

print(f"Algorithm mapping discovered: {cluster_to_label_map}")

# Apply the translation dictionary to our cluster column
df['mapped_prediction'] = df['assigned_cluster'].map(cluster_to_label_map)

print("\n--- Final Exam Grade ---")
# Now we compare the translated predictions to the extracted true labels
final_accuracy = accuracy_score(df[true_label_column], df['mapped_prediction'])

print(f"Real Unsupervised Model Accuracy: {final_accuracy * 100:.2f}%")

1. Extracting the hidden true labels...
Discovered Classes: ['exp' 'pi' 'sum' 'skill' 'edu' 'obj' 'qc' '']

--- The Overlap Grid ---
assigned_cluster     0      1     2     3     4     5      6
true_label                                                  
                     0      0     0     0     0     0      1
edu                  3      0    19     7  8646     9    811
exp               2098      2  5860    46     1   542  32609
obj                  7      0    16  1733     0    46    428
pi                  17  12424    29     5     0    23    795
qc                  13      0    24     4     0    15    921
skill               77      0   144  4721     0     1     31
sum                276      0   552    11     0  5610     93

--- Mapping Clusters to True Labels ---
Algorithm mapping discovered: {0: 'exp', 1: 'pi', 2: 'exp', 3: 'skill', 4: 'edu', 5: 'sum', 6: 'exp'}

--- Final Exam Grade ---
Real Unsupervised Model Accuracy: 91.48%


In [15]:
import joblib
import numpy as np

print("Exporting the AI Brain...")
# Save the TF-IDF Vectorizer (The Vocabulary)
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

# Save the K-Means Model (The Clustering Engine)
joblib.dump(kmeans, 'kmeans_model.pkl')

# Save the Translation Dictionary (The Label Mapper)
joblib.dump(cluster_to_label_map, 'label_map.pkl')

# Save cluster centers for confidence scoring
np.save('cluster_centers.npy', kmeans.cluster_centers_)
joblib.dump('cluster_centers.npy', 'cluster_centers.pkl')

print("Success! Check your folder for the .pkl and .npy files.")

Exporting the AI Brain...
Success! Check your folder for the .pkl and .npy files.


In [16]:
import json

# Save the exact cleaning logic as metadata so the app can replicate it
cleaning_config = {
    "tag_pattern": r'^(Exp|PI|Sum|Edu|Skill|Obj|QC)\t',  # prefix tags used in training data
    "note": "advanced_clean_text was applied during training"
}

with open('cleaning_config.json', 'w') as f:
    json.dump(cleaning_config, f)

print("Saved cleaning_config.json")
print("label_map:", cluster_to_label_map)  # sanity check — paste output here

Saved cleaning_config.json
label_map: {0: 'exp', 1: 'pi', 2: 'exp', 3: 'skill', 4: 'edu', 5: 'sum', 6: 'exp'}
